Szukamy najlepszego r które minimalizuje RMSE

In [1]:
import pandas as pd
import numpy as np
import torch
from modules.train import train_sgd_model_best_r
from notebooks.best import Z_approx_train

ratings = "../data/ratings.csv"

# czy jest różnica w r ze względu na wybór adam czy sgd
best_r_to_sgd_adam = train_sgd_model_best_r(ratings, optimizer_name="adam")
print(best_r_to_sgd_adam)

best_r_to_sgd_sgd = train_sgd_model_best_r(ratings, optimizer_name="sgd")
print(best_r_to_sgd_sgd)


--- Testowanie r = 1 ---
Testowe RMSE dla r=1: 1.7055
--- Testowanie r = 5 ---
Testowe RMSE dla r=5: 1.5102
--- Testowanie r = 10 ---
Testowe RMSE dla r=10: 1.6550
--- Testowanie r = 15 ---
Testowe RMSE dla r=15: 2.0049
--- Testowanie r = 20 ---
Testowe RMSE dla r=20: 2.2121
--- Testowanie r = 30 ---
Testowe RMSE dla r=30: 2.9160
--- Testowanie r = 40 ---
Testowe RMSE dla r=40: 3.4947
--- Testowanie r = 50 ---
Testowe RMSE dla r=50: 4.2697
--- Testowanie r = 100 ---
Testowe RMSE dla r=100: 7.4070
Zakończono! Najlepsze r = 5 z RMSE = 1.5102
5
--- Testowanie r = 1 ---
Testowe RMSE dla r=1: 3.7390
--- Testowanie r = 5 ---
Testowe RMSE dla r=5: 4.2462
--- Testowanie r = 10 ---
Testowe RMSE dla r=10: 4.7557
--- Testowanie r = 15 ---
Testowe RMSE dla r=15: 5.2071
--- Testowanie r = 20 ---
Testowe RMSE dla r=20: 5.6192
--- Testowanie r = 30 ---
Testowe RMSE dla r=30: 6.4253
--- Testowanie r = 40 ---
Testowe RMSE dla r=40: 7.1550
--- Testowanie r = 50 ---
Testowe RMSE dla r=50: 7.8222
--- Test

W przypadku SGD, najpierw sprawdzamy dla r = 1, 5, 10, 15, 20, 30, 40, 50, 100. Widzimy że dla dużych r wraz ze wzrostem r rośnie nam RMSE, początkowo dla optimizer Adam lepszy jest r = 5, dla SGD r = 1.

In [2]:
best_r_to_sgd_adam = train_sgd_model_best_r(ratings, optimizer_name="adam", r_values=[1, 2, 3, 4, 5])
print(best_r_to_sgd_adam)

best_r_to_sgd_sgd = train_sgd_model_best_r(ratings, optimizer_name="sgd", r_values=[1, 2, 3, 4, 5])
print(best_r_to_sgd_sgd)

--- Testowanie r = 1 ---
Testowe RMSE dla r=1: 1.7646
--- Testowanie r = 2 ---
Testowe RMSE dla r=2: 1.4042
--- Testowanie r = 3 ---
Testowe RMSE dla r=3: 1.3688
--- Testowanie r = 4 ---
Testowe RMSE dla r=4: 1.4181
--- Testowanie r = 5 ---
Testowe RMSE dla r=5: 1.5567
Zakończono! Najlepsze r = 3 z RMSE = 1.3688
3
--- Testowanie r = 1 ---
Testowe RMSE dla r=1: 3.7710
--- Testowanie r = 2 ---
Testowe RMSE dla r=2: 3.8661
--- Testowanie r = 3 ---
Testowe RMSE dla r=3: 4.1025
--- Testowanie r = 4 ---
Testowe RMSE dla r=4: 4.1939
--- Testowanie r = 5 ---
Testowe RMSE dla r=5: 4.2438
Zakończono! Najlepsze r = 1 z RMSE = 3.7710
1


Po sprawdzeniu dla r = 1, 2, 3, 4, 5 mamy że dla optimizer Adam najlepszy r = 3, dla optimizer SGD  najlepszy r = 1.

BEST_2

In [ ]:
import pandas as pd
import ipywidgets
import numpy as np
from sklearn.decomposition import NMF, TruncatedSVD
from modules.utils import build_rating_matrix, create_cv_folds, evaluate_fold
from rich.progress import Progress
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import root_mean_squared_error
from modules.train import train_BEST_2_model

path = "../data/ratings.csv"
df = pd.read_csv(path)
folds = create_cv_folds(df, 5, "BEST_2") # tu tworzymy foldy

rs = range(2, 11)
rmse = {}

with Progress() as p:
    t = p.add_task(description = "initialization", total=len(rs)*len(folds), visible=False)
    for r in rs: # tu sprawdzamy dla każdego r
        p.update(t, description=f"Training with r={r}", refresh=True, visible=True)
        r_rmse = []
        for fold in folds: # tu sprawdzamy po wszystkich foldach
            p.update(t, advance=1)
            Z_train = fold['Z_train']
            train_user_map = fold['user_map']
            train_movie_map = fold['movie_map']
            test_df = fold['test_df']

            W, H = train_BEST_2_model(Z_train, r) # tu dopasowujemy model na train

            Z_approx_train = np.dot(W, H)

            # tu obliczamy i dodajemy rmse dla foldu
            r_rmse.append(evaluate_fold(test_df, train_user_map, train_movie_map, Z_approx_train))

        # tu dodajemy srednie rmse dla r
        rmse[r] = np.mean(r_rmse)

min_rmse = min(rmse.values())
best_r = list(rmse.keys())[list(rmse.values()).index(min_rmse)]
print(f"Best r = {best_r} with RMSE = {min_rmse:.4f}")


Output()